In [1]:
# ================================================================
# Seasonal BYM Logistic Model: beta(s, week)
# ================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags, bmat
from pathlib import Path
from tqdm import tqdm
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr
import os



os.chdir(Path.cwd().parent)

In [2]:
# ================================================================
# Seasonal BYM Logistic Model (FAST & STABLE VERSION)
# beta(s, week), week = 1,...,52
# ================================================================

# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

all_y = snow_cleaned_full.drop(index=no_nbs).reset_index(drop=True)

coords = all_y.iloc[:, :2].to_numpy()
y_full = all_y.iloc[:, 2:].to_numpy()   # (S, T), NA allowed

S, TT = y_full.shape
period = 52

print(f"[INFO] S = {S}, T = {TT}")

# ================================================================
# Build adjacency (IDENTICAL to your original code)
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
    crs="EPSG:4326"
)

gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta_rot = np.arctan2(dif[1], dif[0])
R = np.array([
    [np.cos(theta_rot), -np.sin(theta_rot)],
    [np.sin(theta_rot),  np.cos(theta_rot)]
])

rotated = (xy @ R.T) / 1e6
Distances = squareform(pdist(rotated))

Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)

deg = np.array(Omg.sum(axis=1)).flatten()
assert np.all(deg > 0)

D = diags(deg)
prec_icar = D - Omg

# edge list for ICAR quadratic form (PRECOMPUTE ONCE)
rows_e, cols_e = Omg.nonzero()
mask = rows_e < cols_e
ei = rows_e[mask]
ej = cols_e[mask]

# ================================================================
# Observation indexing
# ================================================================
loc = np.where(~np.isnan(y_full))
pairs = np.column_stack(loc)
pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]

row_idx  = pairs[:, 0]
time_idx = pairs[:, 1]
y_obs    = y_full[row_idx, time_idx]

N = len(y_obs)

# ================================================================
# Week index (KEY MODELING CHOICE)
# ================================================================
week_idx = time_idx % 52    # 0,...,51

# ================================================================
# Covariates (4)
# ================================================================
t_raw = time_idx + 1
t_trend = (t_raw - t_raw.mean()) / t_raw.std(ddof=0)

X_cov = np.column_stack([
    np.ones(N),
    np.cos(2*np.pi*t_raw / period),
    np.sin(2*np.pi*t_raw / period),
    t_trend
])

I = 4          # covariates
K = 2 * I      # ICAR + IID

# ================================================================
# MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000
total_iters = burn + tot_save * thin

a_tau = 0.01
b_tau = 0.01
ridge = 1e-6

# ================================================================
# ================================================================
# PRECOMPUTATION (CRITICAL FOR SPEED)
# ================================================================
# ================================================================

# ------------------------------------------------
# Pre-build design matrices for each week
# ------------------------------------------------
print("[INFO] Building design matrices for each week...")

X_week = [None] * 52
y_week = [None] * 52
s_week = [None] * 52

for w in range(52):

    idx = (week_idx == w)
    if not np.any(idx):
        continue

    s_week[w] = row_idx[idx]
    y_week[w] = y_obs[idx]
    Xw_cov = X_cov[idx]
    Nw = Xw_cov.shape[0]

    rows, cols, vals = [], [], []

    for n, s in tqdm(
        list(enumerate(s_week[w])),
        desc=f"Design matrix week {w+1}",
        leave=False
    ):
        for i in range(I):
            rows.append(n)
            cols.append((2*i)*S + s)
            vals.append(Xw_cov[n, i])

            rows.append(n)
            cols.append((2*i+1)*S + s)
            vals.append(Xw_cov[n, i])

    X_week[w] = coo_matrix(
        (vals, (rows, cols)),
        shape=(Nw, K * S)
    ).tocsr()

print("[INFO] Design matrices built.")

# ------------------------------------------------
# Pre-build ICAR / IID block skeleton
# ------------------------------------------------
Q_blocks_base = []
for k in range(K):
    if k % 2 == 0:
        Q_blocks_base.append(prec_icar)              # ICAR
    else:
        Q_blocks_base.append(diags(np.ones(S)))      # IID

# ================================================================
# Storage
# ================================================================
theta_week = [np.zeros(K * S) for _ in range(52)]
tau_week   = [np.ones(K)      for _ in range(52)]

theta_save = np.zeros((52, K*S, tot_save))
tau_save   = np.zeros((52, K,   tot_save))

save_idx = 0

# ================================================================
# MCMC
# ================================================================
print("[INFO] Starting MCMC...")

for it in tqdm(range(total_iters), desc="MCMC"):

    for w in range(52):

        Xmat = X_week[w]
        if Xmat is None:
            continue

        y_w = y_week[w]
        theta = theta_week[w]
        tau   = tau_week[w]

        # ----------------------------
        # Poly-Gamma
        # ----------------------------
        phi = Xmat @ theta
        omega = random_polyagamma(1, np.clip(phi, -20, 20))

        # ----------------------------
        # Posterior precision
        # ----------------------------
        XtOmega = Xmat.T.multiply(omega)

        Prec_prior = bmat(
            [[Q_blocks_base[i] / tau[i] if i == j else None for j in range(K)]
             for i in range(K)],
            format="csr"
        )

        Prec_post = XtOmega @ Xmat + Prec_prior + ridge * diags(np.ones(K*S))

        chol = cholesky(Prec_post)
        mu = chol.solve_A(Xmat.T @ (y_w - 0.5))
        theta = mu + chol.solve_A(np.random.randn(K*S))

        # ----------------------------
        # ICAR identifiability
        # ----------------------------
        for i in range(I):
            sl = slice((2*i)*S, (2*i+1)*S)
            theta[sl] -= theta[sl].mean()

        # ----------------------------
        # Update tau (EDGE-BASED, STABLE)
        # ----------------------------
        for i in range(I):

            sl1 = slice((2*i)*S, (2*i+1)*S)
            b1 = theta[sl1]
            quad1 = np.sum((b1[ei] - b1[ej])**2)

            rate1 = b_tau + quad1 / 2
            tau[2*i] = np.random.gamma(
                a_tau + (S-1)/2,
                scale=1.0 / rate1
            )

            sl2 = slice((2*i+1)*S, (2*i+2)*S)
            b2 = theta[sl2]
            quad2 = b2 @ b2

            rate2 = b_tau + quad2 / 2
            tau[2*i+1] = np.random.gamma(
                a_tau + S/2,
                scale=1.0 / rate2
            )

        theta_week[w] = theta
        tau_week[w]   = tau

    # ----------------------------
    # Save
    # ----------------------------
    if it >= burn and (it - burn) % thin == 0:
        for w in range(52):
            theta_save[w, :, save_idx] = theta_week[w]
            tau_save[w, :, save_idx]   = tau_week[w]
        save_idx += 1
        if save_idx == tot_save:
            break

print("[INFO] MCMC finished.")

# ================================================================
# Save results
# ================================================================
np.savez_compressed(
    r"D:\77\Research\temp\snow\bym_seasonal_week_fast.npz",
    theta=theta_save,
    tau=tau_save
)

print("[INFO] Results saved.")


[INFO] S = 1601, T = 2704
[INFO] Building design matrices for each week...


[INFO] Design matrices built.
[INFO] Starting MCMC...


MCMC:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_14516\1729398997.py:216: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  chol = cholesky(Prec_post)
MCMC: 100%|█████████▉| 5995/6000 [5:08:46<00:15,  3.09s/it]  


[INFO] MCMC finished.
[INFO] Results saved.
